In [ ]:
#| default_exp core

# core

> a record, the file it is written to, and how it reads back

A folder of JSONL files. `ledger/` is the committed summary. `detail/` is machine-local: whole tool arguments, whole outputs, and the hashes of the lines an edit added.

In [ ]:
#| export
import hashlib, os, time, uuid
from pathlib import Path

import orjson
from fastcore.basics import AttrDict, patch
from fastcore.foundation import L
from gheasy.repo import GitRepo

An unknown `kind` is skipped, so a newer writer can add one. `MAX_LINE` caps a committed line.

In [ ]:
#| export
KINDS = ('session', 'step', 'touch', 'commit', 'note')
LEDGER, DETAIL = 'ledger', 'detail'
DIR = '.panjika'
MAX_LINE = 16_000
MAX_DETAIL = 400_000

## One line

In [ ]:
#| export
def dumps(rec):
    "One record as the bytes of a line, newline included."
    return orjson.dumps(rec, option=orjson.OPT_SORT_KEYS) + b'\n'


def loads(line):
    "One line as a record, or `None` when the line is not a JSON object."
    try: rec = orjson.loads(line)
    except orjson.JSONDecodeError: return None
    return AttrDict(rec) if isinstance(rec, dict) else None

In [ ]:
from fastcore.test import test_eq
r = loads(dumps({'kind': 'note', 'text': 'hello'}))
test_eq(r.kind, 'note')
test_eq(loads(b'not json'), None)
test_eq(loads(b'[1,2]'), None)

In [ ]:
#| export
def now(): return round(time.time(), 3)


def new_id(prefix=''):
    "A short unique id. Records carry one so a union merge can be deduplicated."
    return f'{prefix}{uuid.uuid4().hex[:12]}'


def hashed(data, n=16):
    "A short content hash, for file contents and for the text of a changed line."
    if isinstance(data, str): data = data.encode('utf-8', 'replace')
    return hashlib.blake2b(data, digest_size=n // 2).hexdigest()


def file_hash(path):
    "The hash of a file's bytes, or `''` when it is not there."
    try: return hashed(Path(path).read_bytes())
    except OSError: return ''

In [ ]:
test_eq(hashed('abc'), hashed('abc'))
assert hashed('abc') != hashed('abd')
test_eq(len(hashed('abc')), 16)
test_eq(file_hash('/does/not/exist'), '')

## Where a ledger lives

Walk up from `start` for a ledger, then the repository root, else `~`. Every harness in a repository lands on one file.

In [ ]:
#| export
def git_root(start='.'):
    "The repository `start` is inside, or `None`."
    try:
        from gheasy.repo import repo_root
        root = repo_root(str(start))
        return None if root is None else Path(root)
    except Exception: return None


def find_home(start='.'):
    "The ledger folder for `start`: an existing one above it, else its repository, else `~`."
    here = Path(start).resolve()
    for d in (here, *here.parents):
        if (d/DIR).is_dir(): return d/DIR
    root = git_root(here)
    return (root or Path.home())/DIR

## The folder

Sharded by month. `init` writes `merge=union` for the ledger tier and gitignores `detail/`.

In [ ]:
#| export
GITATTRIBUTES = """# Append-only. Take both sides of a merge; readers deduplicate by record id.
ledger/*.jsonl merge=union
"""

GITIGNORE = """# Machine-local: whole tool arguments, whole outputs, exact changed lines.
detail/
"""


class Home:
    "One ledger folder, and the shards inside it."

    def __init__(self, path=None, start='.'):
        self.path = Path(path) if path else find_home(start)
        self.root = self.path.parent

    def __repr__(self): return f'Home({self.path})'
    def __eq__(self, other): return isinstance(other, Home) and self.path == other.path

    def dir(self, tier): return self.path/tier

    def shard(self, tier=LEDGER, at=None):
        "The file a record written at `at` belongs in."
        stamp = time.strftime('%Y-%m', time.localtime(at if at else now()))
        return self.dir(tier)/f'{stamp}.jsonl'

    def shards(self, tier=LEDGER):
        "Every shard of one tier, oldest first."
        d = self.dir(tier)
        return sorted(d.glob('*.jsonl')) if d.is_dir() else []

    @property
    def exists(self): return self.path.is_dir()

    def init(self):
        "Make the folder and the two dotfiles that let the ledger tier be committed. Idempotent."
        for tier in (LEDGER, DETAIL): self.dir(tier).mkdir(parents=True, exist_ok=True)
        (self.path/'.gitattributes').write_text(GITATTRIBUTES)
        (self.path/'.gitignore').write_text(GITIGNORE)
        return self

## Appending

One record, one `O_APPEND` write, so two harnesses never interleave. Too long is clipped, longest field first.

In [ ]:
#| export
def clip(rec, limit):
    "Shorten a record until its line fits, longest string field first."
    line = dumps(rec)
    if len(line) <= limit: return rec, len(line)
    rec = dict(rec)
    for k in sorted((k for k, v in rec.items() if isinstance(v, str)),
                    key=lambda k: len(rec[k]), reverse=True):
        over = len(dumps(rec)) - limit
        if over <= 0: break
        keep = max(0, len(rec[k]) - over - 32)
        rec[k] = rec[k][:keep] + f'... [clipped {len(rec[k]) - keep} chars]' if keep else ''
    return rec, len(dumps(rec))


def append(path, rec, limit=MAX_LINE):
    "Write one record as one line. Returns the record as it was written."
    rec, _ = clip(rec, limit)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o644)
    try: os.write(fd, dumps(rec))
    finally: os.close(fd)
    return AttrDict(rec)

In [ ]:
import tempfile
d = Path(tempfile.mkdtemp())
h = Home(d/'.panjika').init()
test_eq((h.path/'.gitattributes').read_text().splitlines()[-1], 'ledger/*.jsonl merge=union')
assert 'detail/' in (h.path/'.gitignore').read_text()
append(h.shard(), {'kind': 'note', 'id': 'n1', 'at': now(), 'text': 'first'})
append(h.shard(), {'kind': 'note', 'id': 'n2', 'at': now(), 'text': 'second'})
test_eq(len(h.shard().read_text().strip().splitlines()), 2)

In [ ]:
big, n = clip({'kind': 'step', 'id': 's1', 'tool': 'Edit', 'summary': 'x' * 40_000}, MAX_LINE)
assert n <= MAX_LINE
test_eq(big['tool'], 'Edit')
assert big['summary'].endswith('chars]')

## Reading back

Oldest first, first record kept per `id`. That dedupe is what makes `merge=union` safe.

In [ ]:
#| export
def read_shard(path):
    "Every readable record in one shard."
    out = L()
    try: raw = Path(path).read_bytes()
    except OSError: return out
    for line in raw.splitlines():
        if not line.strip(): continue
        rec = loads(line)
        if rec is not None and rec.get('kind') in KINDS and rec.get('id'): out.append(rec)
    return out


def records(home, tier=LEDGER):
    "Every record in a tier, oldest first, deduplicated by id."
    seen, out = set(), L()
    for shard in home.shards(tier):
        for rec in read_shard(shard):
            if rec.id in seen: continue
            seen.add(rec.id)
            out.append(rec)
    return out.sorted(key=lambda r: r.get('at') or 0)

In [ ]:
dup = {'kind': 'note', 'id': 'n1', 'at': now(), 'text': 'first'}
append(h.shard(), dup)
test_eq(len(records(h)), 2)
test_eq(sorted(r.id for r in records(h)), ['n1', 'n2'])

with h.shard().open('a') as f: f.write('{ this is not json\n')
append(h.shard(), {'kind': 'note', 'id': 'n3', 'at': now(), 'text': 'third'})
test_eq(len(records(h)), 3)

append(h.shard(), {'kind': 'something-new', 'id': 'n4', 'at': now()})
test_eq(sorted(r.id for r in records(h)), ['n1', 'n2', 'n3'])

Two branches both append, then merge. `merge=union` takes both sides.

In [ ]:
import subprocess

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

g = Path(tempfile.mkdtemp())/'proj'; g.mkdir(parents=True)
_git(g, 'init', '-q', '-b', 'main')
_git(g, 'config', 'user.email', 'a@b.c'); _git(g, 'config', 'user.name', 'Sam')
gh = Home(g/DIR).init()
append(gh.shard(), {'kind': 'note', 'id': 'base', 'at': 1, 'text': 'base'})
_git(g, 'add', '-A'); _git(g, 'commit', '-qm', 'base')

_git(g, 'checkout', '-q', '-b', 'side')
append(gh.shard(), {'kind': 'note', 'id': 'side', 'at': 3, 'text': 'from the side branch'})
_git(g, 'commit', '-aqm', 'side')

_git(g, 'checkout', '-q', 'main')
append(gh.shard(), {'kind': 'note', 'id': 'main', 'at': 2, 'text': 'from main'})
_git(g, 'commit', '-aqm', 'main')

_git(g, 'merge', '-q', '--no-edit', 'side')
test_eq([r.id for r in records(gh)], ['base', 'main', 'side'])

## Folding

A session is every record carrying its id, folded in time order. Later wins, a blank never erases, and `first` fields keep their earliest.

In [ ]:
#| export
def fold(recs, key='session', first=()):
    "Merge records sharing `key` into one row each, later values winning and `first` fields keeping their earliest."
    out, held = {}, {}
    for rec in recs:
        k = rec.get(key)
        if not k: continue
        row = out.setdefault(k, AttrDict(rec))
        for name, v in rec.items():
            if v is None or v == '' or name == 'id': continue
            if name in first:
                if (k, name) in held: continue
                held[(k, name)] = True
            row[name] = v
    return L(out.values())

In [ ]:
rows = fold([AttrDict(kind='session', id='r1', session='s', at=1, status='open', model='opus'),
             AttrDict(kind='session', id='r2', session='s', at=2, status='done', title='a title', model='')])
test_eq(len(rows), 1)
test_eq(rows[0].status, 'done')
test_eq(rows[0].title, 'a title')
test_eq(rows[0].model, 'opus')

held = fold([AttrDict(kind='session', id='r1', session='s', at=1, prompt='the first ask'),
             AttrDict(kind='session', id='r2', session='s', at=2, prompt='a later ask')],
            first=('prompt',))
test_eq(held[0].prompt, 'the first ask')

## What a tool did

`action_for` sorts a tool name into one of `ACTIONS`, first match winning.

In [ ]:
#| export
ACTION_HINTS = (
    ('write', ('edit', 'write', 'patch', 'create', 'apply', 'update', 'insert', 'replace', 'delete')),
    ('run',   ('bash', 'shell', 'exec', 'run', 'terminal', 'command', 'python', 'notebook_run')),
    ('net',   ('web', 'fetch', 'url', 'browser', 'http', 'research', 'crawl')),
    ('search',('grep', 'search', 'glob', 'find', 'index', 'similar', 'outline', 'symbols')),
    ('read',  ('read', 'view', 'cat', 'open', 'ls', 'list', 'show', 'inspect', 'vars')),
    ('plan',  ('todo', 'plan', 'task')),
    ('ask',   ('ask', 'approval', 'question', 'permission')),
)

ACTIONS = ('write', 'run', 'net', 'search', 'read', 'plan', 'ask', 'other')


def action_for(tool):
    "Which of `ACTIONS` a tool name means. `other` when nothing matches."
    name = str(tool or '').lower()
    for action, hints in ACTION_HINTS:
        if any(h in name for h in hints): return action
    return 'other'

In [ ]:
from fastcore.test import test_eq
test_eq(action_for('Edit'), 'write')
test_eq(action_for('apply_patch'), 'write')
test_eq(action_for('Bash'), 'run')
test_eq(action_for('WebFetch'), 'net')
test_eq(action_for('Grep'), 'search')
test_eq(action_for('Read'), 'read')
test_eq(action_for('mcp__something__weird'), 'other')

## The scribe

Every record carries an `id`, used only to drop a duplicate. A backfill passes its own, so a re-read changes nothing.

In [ ]:
#| export
def _target(value, n=160):
    "The one-line target of a step: a path, a command, a url, whatever it was pointed at."
    return ' '.join(str(value or '').split())[:n]


class Scribe:
    "Appends to one ledger."

    def __init__(self,
                 home=None,         # the ledger folder. None finds one from `start`
                 session='',        # the session every record joins. None starts a fresh id
                 start='.',         # where to look for a ledger, and the repository to describe
                 detail=True):      # write the machine-local tier as well as the committed one
        self.home = home if isinstance(home, Home) else Home(home, start)
        self.session = str(session or new_id('s-'))
        self.start, self.detail = str(start), bool(detail)
        self._seq = 0

    def __repr__(self): return f'Scribe({self.session} -> {self.home.path})'

    def write(self, kind, detail=None, **fields):
        "Append one record. `detail` goes to the machine-local tier under the same id."
        rec = {'kind': kind, 'id': fields.pop('id', None) or new_id(),
               'at': fields.pop('at', None) or now(),
               'session': fields.pop('session', None) or self.session, **fields}
        rec = {k: v for k, v in rec.items() if v is not None}
        out = append(self.home.shard(LEDGER, rec['at']), rec)
        if detail and self.detail:
            append(self.home.shard(DETAIL, rec['at']),
                   {'kind': kind, 'id': rec['id'], 'at': rec['at'], 'session': rec['session'], **detail},
                   MAX_DETAIL)
        return out

In [ ]:
#| export
def repo_facts(start='.'):
    "The repository name, root and current branch for `start`, as far as they can be told."
    root = git_root(start)
    branch = ''
    if root is not None:
        try: branch = GitRepo(root).run('rev-parse', '--abbrev-ref', 'HEAD').strip()
        except Exception: branch = ''
    return {'repo': root.name if root else '', 'root': str(root) if root else '',
            'branch': '' if branch == 'HEAD' else branch}

In [ ]:
#| export
@patch
def begin(self:Scribe, harness, model='', prompt='', title='', parent='', agent='', **fields):
    "Record that a session started. Returns its id."
    self.write('session', harness=str(harness), model=str(model or ''),
               prompt=_target(prompt, 2000), title=str(title or ''), status='open',
               parent=str(parent or ''), agent=str(agent or ''),
               started=fields.pop('started', None) or fields.get('at') or now(),
               cwd=str(Path(self.start).resolve()), **repo_facts(self.start), **fields)
    return self.session


@patch
def end(self:Scribe, status='done', **fields):
    "Record that a session finished, with whatever counters the harness can supply."
    return self.write('session', status=str(status),
                      ended=fields.pop('ended', None) or fields.get('at') or now(), **fields)

`step` records one tool call, `note` a line of prose.

In [ ]:
#| export
@patch
def step(self:Scribe, tool, target='', ok=True, secs=0.0, action='', summary='',
         args=None, output=None, **fields):
    "Record one tool call. Returns the step id."
    self._seq += 1
    detail = None
    if args is not None or output is not None:
        detail = {'tool': str(tool), 'args': args, 'output': None if output is None else str(output)}
    rec = self.write('step', detail=detail, tool=str(tool), seq=self._seq,
                     action=action or action_for(tool), target=_target(target),
                     ok=bool(ok), secs=round(float(secs or 0), 3),
                     summary=_target(summary, 300), **fields)
    return rec.id


@patch
def note(self:Scribe, text, **fields):
    "Record a line of prose against the session: a plan, a summary, a reason."
    return self.write('note', text=str(text)[:4000], **fields).id

A hook reads a diff, a backfill reads the two bodies. Both must hash the same strings.

In [ ]:
#| export
def text_changes(before, after):
    "The lines `after` adds to `before` and the ones it drops, for a change git cannot be asked about."
    import difflib
    b, a = (before or '').splitlines(), (after or '').splitlines()
    add, rem = [], []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, b, a).get_opcodes():
        if tag in ('replace', 'delete'): rem += b[i1:i2]
        if tag in ('replace', 'insert'): add += a[j1:j2]
    return add, rem


def diff_lines(diff):
    "The added and removed lines of a unified diff, without its headers."
    added, removed = [], []
    for line in str(diff or '').splitlines():
        if line[:4] in ('+++ ', '--- ') or line in ('+++', '---'): continue
        if line.startswith('+'): added.append(line[1:])
        elif line.startswith('-'): removed.append(line[1:])
    return added, removed


def body_lines(path):
    "Every line of a file, for a change git will not diff because it is not tracking it."
    try: return Path(path).read_text(encoding='utf-8', errors='replace').splitlines()
    except OSError: return []


def head_hash(root, path):
    "The hash of a path's committed content at HEAD, or `''` when it is not committed."
    try: return hashed(GitRepo(root).run('show', f'HEAD:{path}'))
    except Exception: return ''

In [ ]:
from fastcore.test import test_eq

before = 'int x;\n'
after = 'int x;\n++counter;\n\tindented = 3\ntrailing = 4   \n'
diff = ('--- a/f.c\n+++ b/f.c\n@@ -1,1 +1,4 @@\n int x;\n'
        '+++counter;\n+\tindented = 3\n+trailing = 4   \n')
test_eq(diff_lines(diff)[0], text_changes(before, after)[0])
test_eq(diff_lines(diff)[0], ['++counter;', '\tindented = 3', 'trailing = 4   '])

`touch` measures against `HEAD`, so the last touch is the whole net change. git will not diff an untracked file, so a created file has its whole body read.

In [ ]:
#| export
@patch
def touch(self:Scribe,
          path,               # the file that moved, absolute or relative to `start`
          action='edit',      # what happened to it: edit, create, delete
          step='',            # the id of the step that did it
          before=None,        # the body before the change, for a backfill
          after_text=None,    # the body after it, for a backfill
          **fields):
    "Record that a file was touched, and how it now differs from `HEAD`."
    p = Path(path)
    root = git_root(p if p.is_absolute() else self.start)
    rel = str(p.resolve().relative_to(root)) if root and p.is_absolute() else str(path)
    ranges, added, removed, tracked = [], [], [], bool(root)
    if after_text is not None:
        added, removed = text_changes(before, after_text)
    elif root is not None:
        try:
            ch = GitRepo(root).file_changes(rel)
            ranges = [[r['from'], r['to'], r['kind']] for r in ch.get('ranges') or []]
            added, removed = diff_lines(ch.get('diff'))
            tracked = not (ch.get('status') or {}).get('untracked', False)
        except Exception: pass
    if not added and not tracked and root is not None:
        added = body_lines(p if p.is_absolute() else Path(self.start)/path)
    detail = {'path': rel, 'lines': [hashed(t) for t in added][:2000]} if added else None
    rec = self.write('touch', detail=detail, path=rel, action=str(action), step=str(step or ''),
                     head=head_hash(root, rel) if root else '',
                     after=(hashed(after_text) if after_text is not None else
                            file_hash(p if p.is_absolute() else Path(self.start)/path)),
                     added=len(added), removed=len(removed), ranges=ranges[:200],
                     tracked=tracked, **fields)
    return rec.id


@patch
def commit(self:Scribe, sha, subject='', author='', files=(), branch='', **fields):
    "Record that a commit exists and which session it belongs to."
    return self.write('commit', sha=str(sha), subject=_target(subject, 300), author=str(author or ''),
                      files=list(files)[:500], branch=str(branch or ''), **fields).id

A whole session, written and read back.

In [ ]:
import subprocess, tempfile

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('def add(a, b):\n    return a + b\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')

sc = Scribe(home=d/'.panjika', start=d)
sc.home.init()
sc.begin('claude-code', model='opus-5', prompt='make add handle strings')
(d/'app.py').write_text('def add(a, b):\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
sid = sc.session
step = sc.step('Edit', target='app.py', secs=0.4, args={'file_path': 'app.py'})
sc.touch(d/'app.py', 'edit', step)
sc.end('done', turns=1, steps_ok=1, steps_fail=0)

rows = {r.kind: r for r in records(sc.home)}
test_eq(sorted(rows), ['session', 'step', 'touch'])
test_eq(rows['touch'].path, 'app.py')
test_eq(rows['touch'].added, 1)
test_eq(fold(records(sc.home).filter(lambda r: r.kind == 'session'))[0].status, 'done')

The committed tier carries no source. The machine-local tier carries the line hashes.

In [ ]:
ledger_text = sc.home.shard().read_text()
assert 'isinstance' not in ledger_text, 'the ledger tier must not carry source'
detail = records(sc.home, DETAIL).filter(lambda r: r.kind == 'touch')
test_eq(len(detail[0].lines), 1)

## Reading

Rereads a tier when a shard changes on disk.

In [ ]:
#| export
def shard_stamp(home, tier=LEDGER):
    "What changes when a shard does. The cache key."
    out = []
    for p in home.shards(tier):
        try: st = p.stat()
        except OSError: continue
        out.append((p.name, st.st_size, st.st_mtime_ns))
    return tuple(out)


class Ledger:
    "One ledger, read. Rereads a tier when a shard on disk has changed."

    def __init__(self, home=None, start='.'):
        self.home = home if isinstance(home, Home) else Home(home, start)
        self._cache, self._stamps = {}, {}

    def __repr__(self): return f'Ledger({self.home.path}, {len(self.all())} records)'

    @property
    def exists(self): return self.home.exists

    def all(self, tier=LEDGER):
        "Every record in a tier, oldest first."
        stamp = shard_stamp(self.home, tier)
        if self._stamps.get(tier) != stamp:
            self._cache[tier], self._stamps[tier] = records(self.home, tier), stamp
        return self._cache[tier]

    def of(self, kind, tier=LEDGER):
        "Every record of one kind."
        return self.all(tier).filter(lambda r: r.kind == kind)

`OPENING` fields keep their *first* value: the opening ask, and the start. `headline` skips prompts the harness injected. Counters are computed on read, because a hook cannot total.

In [ ]:
#| export
OPENING = ('prompt', 'started')


def asks(recs, sid):
    "Every prompt put to one session, in order, each with where it came from."
    return [(r.prompt, r.get('origin') or 'human') for r in recs
            if r.get('session') == sid and r.get('prompt')]


def headline(asked):
    "What a session was for: the first thing a person asked it."
    return next((p for p, o in asked if o != 'injected'), asked[0][0] if asked else '')


def counters(steps):
    "What a session's steps add up to: how many worked, how many did not, and which tools."
    ok = sum(1 for s in steps if s.get('ok', True))
    tools, actions = {}, {}
    for s in steps:
        tools[s.get('tool', '?')] = tools.get(s.get('tool', '?'), 0) + 1
        actions[s.get('action', 'other')] = actions.get(s.get('action', 'other'), 0) + 1
    return {'n_steps': len(steps), 'steps_ok': ok, 'steps_fail': len(steps) - ok,
            'secs': round(sum(float(s.get('secs') or 0) for s in steps), 2),
            'tools': dict(sorted(tools.items(), key=lambda kv: -kv[1])),
            'actions': dict(sorted(actions.items(), key=lambda kv: -kv[1]))}


def _since(value):
    "Seconds since the epoch for `7d`, `12h`, `30m`, a count of days, or a timestamp."
    if value in (None, ''): return 0
    if isinstance(value, (int, float)) and value > 10_000_000: return float(value)
    text = str(value).strip().lower()
    units = {'d': 86400, 'h': 3600, 'm': 60, 'w': 604800, 's': 1}
    n, unit = (text[:-1], text[-1]) if text[-1:] in units else (text, 'd')
    try: return time.time() - float(n) * units[unit]
    except ValueError: return 0

In [ ]:
#| export
@patch
def sessions(self:Ledger, limit=20, harness='', since='', repo='', path='', status=''):
    "Session rows, newest first. Every filter is optional and they compose."
    recs = self.of('session')
    rows = fold(recs, first=OPENING)
    for r in rows:
        if (asked := asks(recs, r.session)): r.prompt = headline(asked)
    if path: rows = rows.filter(lambda r: r.session in self.sessions_touching(path))
    for field, want in (('harness', harness), ('repo', repo), ('status', status)):
        if want: rows = rows.filter(lambda r, f=field, w=want: r.get(f) == w)
    if since:
        after = _since(since)
        rows = rows.filter(lambda r: (r.get('started') or r.get('at') or 0) >= after)
    rows = rows.sorted(key=lambda r: r.get('started') or r.get('at') or 0, reverse=True)
    return rows if not limit else rows[:int(limit)]


@patch
def sessions_touching(self:Ledger, path):
    "The ids of every session that touched `path`."
    p = str(path)
    return {r.session for r in self.of('touch') if r.get('path') == p}

Five edits to one file leave five touches of the same net change, so the last is the whole of it.

In [ ]:
#| export
@patch
def files(self:Ledger, session):
    "What one session did to each file it touched, one row per path."
    out = {}
    for r in self.of('touch'):
        if r.session == session: out[r.path] = r
    return L(out.values()).sorted(key=lambda r: r.path)


@patch
def session(self:Ledger, sid):
    "One session, with everything recorded against it."
    recs = self.of('session')
    rows = fold(recs, first=OPENING).filter(lambda r: r.session == sid)
    row = AttrDict(rows[0]) if rows else AttrDict(kind='session', session=sid, id=sid)
    asked = asks(recs, sid)
    row['prompts'] = L(asked)
    row['prompt'] = headline(asked) if asked else row.get('prompt', '')
    row['last_prompt'] = asked[-1][0] if asked else ''
    for kind in ('step', 'touch', 'commit', 'note'):
        row[f'{kind}s'] = self.of(kind).filter(lambda r: r.session == sid).sorted(
            key=lambda r: r.get('at') or 0)
    row['files'] = self.files(sid)
    row.update(counters(row.steps))
    began = row.get('started') or row.get('at') or 0
    row['seconds'] = round((row.get('ended') or row.get('at') or 0) - began, 1) if began else 0.0
    return row

## The trail over a file

In [ ]:
#| export
@patch
def trail(self:Ledger, path, limit=50):
    "Every session that touched `path`, newest first, with its touch of that file."
    p, out = str(path), []
    by_session = {}
    for r in self.of('touch'):
        if r.get('path') == p: by_session[r.session] = r
    rows = {r.session: r for r in fold(self.of('session'))}
    commits = {}
    for c in self.of('commit'):
        if p in (c.get('files') or ()): commits.setdefault(c.session, []).append(c)
    for sid, touch in by_session.items():
        row = AttrDict(rows.get(sid) or {'kind': 'session', 'session': sid, 'id': sid})
        out.append(AttrDict(row, touch=touch, commits=L(commits.get(sid) or []),
                            at=touch.get('at') or row.get('at') or 0))
    out = L(out).sorted(key=lambda r: r.at, reverse=True)
    return out if not limit else out[:int(limit)]


@patch
def touched(self:Ledger, since='', limit=200):
    "Every path the ledger knows about, most recently touched first."
    after, seen = _since(since), {}
    for r in self.of('touch'):
        if (r.get('at') or 0) < after: continue
        seen[r.path] = max(seen.get(r.path, 0), r.get('at') or 0)
    return L(sorted(seen.items(), key=lambda kv: -kv[1])[:int(limit)])

In [ ]:
#| export
@patch
def search(self:Ledger, query, limit=20):
    "Sessions whose prompt, title or touched paths mention `query`."
    q = str(query).lower()
    hits = {r.session for r in self.of('touch') if q in str(r.get('path', '')).lower()}
    out = []
    for row in fold(self.of('session')):
        blob = ' '.join(str(row.get(k, '')) for k in ('prompt', 'title', 'model', 'harness'))
        if q in blob.lower() or row.session in hits: out.append(row)
    return L(out).sorted(key=lambda r: r.get('started') or 0, reverse=True)[:int(limit)]


@patch
def detail(self:Ledger, record_id):
    "The machine-local half of one record: whole arguments, whole output, exact changed lines."
    for r in self.all(DETAIL):
        if r.id == record_id: return r
    return None


@patch
def stats(self:Ledger):
    "What this ledger holds."
    kinds = {}
    for r in self.all(): kinds[r.kind] = kinds.get(r.kind, 0) + 1
    rows = fold(self.of('session'))
    return AttrDict(path=str(self.home.path), records=len(self.all()), sessions=len(rows),
                    harnesses=sorted({r.get('harness', '') for r in rows} - {''}),
                    files=len({r.path for r in self.of('touch')}),
                    detail=len(self.all(DETAIL)), **kinds)

Three harnesses, one ledger.

In [ ]:
import subprocess, tempfile
from fastcore.test import test_eq

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('def add(a, b):\n    return a + b\n')
(d/'util.py').write_text('X = 1\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')

def a_session(harness, prompt, path, text):
    sc = Scribe(home=d/'.panjika', start=d)
    sc.home.init()
    sc.begin(harness, model='opus-5', prompt=prompt)
    (d/path).write_text(text)
    sc.touch(d/path, 'edit', sc.step('Edit', target=path, secs=0.2))
    sc.end('done', turns=1)
    return sc.session

one = a_session('claude-code', 'handle strings', 'app.py',
                'def add(a, b):\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
two = a_session('codex', 'add a docstring', 'app.py',
                'def add(a, b):\n    "Add two things."\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
three = a_session('ramabana', 'bump the constant', 'util.py', 'X = 2\n')

led = Ledger(d/'.panjika')
test_eq(len(led.sessions()), 3)
test_eq([r.harness for r in led.sessions()], ['ramabana', 'codex', 'claude-code'])

In [ ]:
rows = led.trail('app.py')
test_eq([r.harness for r in rows], ['codex', 'claude-code'])
test_eq([r.prompt for r in rows], ['add a docstring', 'handle strings'])
test_eq(rows[0].touch.path, 'app.py')

In [ ]:
test_eq([p for p, _ in led.touched()], ['util.py', 'app.py'])
test_eq(len(led.sessions(harness='codex')), 1)
test_eq(len(led.sessions(path='util.py')), 1)
test_eq([r.harness for r in led.search('util')], ['ramabana'])
test_eq(led.session(one).files[0].path, 'app.py')
test_eq(led.stats().files, 2)

row = led.session(one)
test_eq((row.n_steps, row.steps_ok, row.steps_fail), (1, 1, 0))
test_eq(row.tools, {'Edit': 1})
test_eq([s.tool for s in row.steps], ['Edit'])

The opening ask headlines the session, not the last thing said to it.

In [ ]:
sc = Scribe(home=d/'.panjika', start=d)
sc.begin('claude-code', model='opus-5')
sc.write('session', prompt='<task-notification>a subagent finished', origin='injected')
sc.write('session', prompt='port the parser to the new API', origin='human')
sc.write('session', prompt='now fix the lint', origin='human')
sc.end('done')

row = Ledger(d/'.panjika').session(sc.session)
test_eq(row.prompt, 'port the parser to the new API')
test_eq(row.last_prompt, 'now fix the lint')
test_eq(len(row.prompts), 3)
test_eq((row.status, row.model), ('done', 'opus-5'))
test_eq(Ledger(d/'.panjika').sessions()[0].prompt, 'port the parser to the new API')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()